# 🏠 Forecasting Household Energy Consumption
## Notebook 2 — Feature Engineering & Machine Learning

**Capstone Project | UMBC Data Science Master's Program**
Instructor: Dr. Chaojie (Jay) Wang | Author: Kushal Erramilli

---

### What This Notebook Does
Takes the clean hourly dataset from Notebook 1 and answers:

1. **Which features drive prediction?** — lag structure, rolling statistics, calendar
2. **Which model architecture works best?** — Baseline → Ridge → RF → XGBoost → LightGBM → LSTM
3. **How reliable are the predictions?** — walk-forward CV, error by hour and season
4. **What can a homeowner do with this?** — actionable interpretation of results

### Leakage Prevention Policy
All features represent information available *strictly before* the target hour `t`:

| Feature type | Correct | Incorrect (leakage) |
|---|---|---|
| Lag | `GAP[t-1]` → `shift(1)` | `GAP[t]` (no shift) |
| Rolling | `shift(1).rolling(w).mean()` | `.rolling(w).mean()` (includes t) |
| Diff | `shift(1).diff(k)` | `.diff(k)` without prior shift |

**Prerequisite:** Run `01_EDA.ipynb` first to generate `../data/hourly_clean.csv`.

### Notebook Roadmap
| Step | Section |
|------|---------|
| 1 | Load hourly dataset |
| 2 | Feature engineering (47 features, no leakage) |
| 3 | Train / Test split (80/20 temporal) |
| 4 | Evaluation metrics |
| 5 | Naïve baseline (lag-24h) |
| 6 | Ridge Regression |
| 7 | Random Forest |
| 8 | XGBoost |
| 9 | LightGBM |
| 10 | LSTM (Multivariate) |
| 11 | Model comparison |
| 12 | Feature importance — RF / XGB / LGB |
| 13 | Residual analysis |
| 14 | Walk-forward cross-validation |
| 15 | Error by hour and season |
| 16 | Quantile performance analysis |
| 17 | Save models |
| 18 | Final conclusions & recommendations |


## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

import warnings, joblib, os
warnings.filterwarnings("ignore")

SEED       = 42
TEST_RATIO = 0.20
TARGET     = "Global_active_power"
BAR_BLUE   = "#1565C0"
np.random.seed(SEED)
os.makedirs("../models", exist_ok=True)

print("✅ Imports OK")


## 1. Load Hourly Dataset

**Why hourly?** Minute-level data (2 M rows) is computationally expensive
and introduces high-frequency noise that adds no forecasting value for
hour-ahead predictions. Hourly aggregation smooths sensor jitter while
preserving every meaningful temporal pattern identified in Notebook 1.


In [ ]:
df = pd.read_csv("../data/hourly_clean.csv",
                 index_col="Datetime", parse_dates=True)
df.sort_index(inplace=True)

print(f"Shape      : {df.shape}")
print(f"Date range : {df.index.min()}  →  {df.index.max()}")
print(f"Missing    : {df.isnull().sum().sum()}")
df.describe().T


## 2. Feature Engineering

We create **47 features** across five categories. Every feature respects
the leakage policy — no information from hour `t` leaks into the inputs.

### Why each group matters

| Category | Features | Purpose |
|----------|---------|---------|
| **Calendar** | hour, dow, month, quarter, weekend | Captures routine human behaviour cycles |
| **Cyclical** | sin/cos of hour, dow, month | Prevents discontinuity at midnight, Sunday→Monday, Dec→Jan |
| **Lag** | 1, 2, 3, 6, 12, 24, 48, 72, 168, 336 h | Autocorrelation structure from EDA (ACF significant to 2 weeks) |
| **Rolling** | mean, std, min, max over 3, 6, 12, 24, 48, 168 h | Smoothed trend + volatility signal |
| **Diff** | 1 h and 24 h change | Rate of change — is consumption accelerating? |

**Sanity check:** After engineering we verify that no feature has correlation
|r| > 0.98 with the target (would indicate leakage).


In [ ]:
df_feat = df[[TARGET]].copy()

# ── Calendar & cyclical ───────────────────────────────────────────────────
def add_time_features(d):
    d["hour"]      = d.index.hour
    d["dayofweek"] = d.index.dayofweek
    d["month"]     = d.index.month
    d["quarter"]   = d.index.quarter
    d["is_weekend"]= (d.index.dayofweek >= 5).astype(int)
    d["hour_sin"]  = np.sin(2*np.pi*d["hour"]      / 24)
    d["hour_cos"]  = np.cos(2*np.pi*d["hour"]      / 24)
    d["dow_sin"]   = np.sin(2*np.pi*d["dayofweek"] / 7)
    d["dow_cos"]   = np.cos(2*np.pi*d["dayofweek"] / 7)
    d["month_sin"] = np.sin(2*np.pi*d["month"]     / 12)
    d["month_cos"] = np.cos(2*np.pi*d["month"]     / 12)
    return d

# ── Lag (all shift ≥ 1) ──────────────────────────────────────────────────
def add_lag_features(d, lags=[1,2,3,6,12,24,48,72,168,336]):
    for lag in lags:
        d[f"lag_{lag}h"] = d[TARGET].shift(lag)
    return d

# ── Rolling (shift(1) before rolling) ────────────────────────────────────
def add_rolling_features(d, windows=[3,6,12,24,48,168]):
    past = d[TARGET].shift(1)
    for w in windows:
        r = past.rolling(w)
        d[f"roll_mean_{w}h"] = r.mean()
        d[f"roll_std_{w}h"]  = r.std()
        d[f"roll_min_{w}h"]  = r.min()
        d[f"roll_max_{w}h"]  = r.max()
    return d

# ── Diff (shift(1) FIRST — past-to-past change only) ─────────────────────
def add_diff_features(d):
    past = d[TARGET].shift(1)
    d["diff_1h"]  = past.diff(1)
    d["diff_24h"] = past.diff(24)
    return d

df_feat = add_time_features(df_feat)
df_feat = add_lag_features(df_feat)
df_feat = add_rolling_features(df_feat)
df_feat = add_diff_features(df_feat)
df_feat.dropna(inplace=True)

feature_cols = [c for c in df_feat.columns if c != TARGET]
print(f"Rows after dropna : {len(df_feat):,}")
print(f"Feature count     : {len(feature_cols)}")


In [ ]:
# Leakage sanity check
corr_target = df_feat.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("Top 5 feature correlations with target (must all be < 0.98):")
print(corr_target.head())
assert corr_target.max() < 0.98, "⚠️  Leakage detected!"
print(f"\n✅ Leakage check passed — max |r| = {corr_target.max():.4f}")
print(f"   Top predictor: lag_1h (r={corr_target['lag_1h']:.4f})")


## 3. Train / Test Split — 80 / 20 Temporal

**Why temporal (not random) split?**
Random splitting would mix future data into training — a form of leakage.
A temporal split ensures the model is evaluated on data it has *never seen*
chronologically, mimicking real deployment conditions.

**What this means:**
- **Train:** Dec 2006 → Feb 2010 (~3.1 years)
- **Test:**  Feb 2010 → Nov 2010 (~9 months)

The test set spans all four seasons, so seasonal performance differences
measured in Section 15 are meaningful.


In [ ]:
X = df_feat[feature_cols]
y = df_feat[TARGET]

split_idx = int(len(df_feat) * (1 - TEST_RATIO))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape[0]:,} hours  "
      f"({X_train.index.min().date()} → {X_train.index.max().date()})")
print(f"Test : {X_test.shape[0]:,} hours   "
      f"({X_test.index.min().date()} → {X_test.index.max().date()})")

fig = go.Figure()
fig.add_trace(go.Scatter(x=y_train.index, y=y_train, name="Train",
                          line=dict(color="#1565C0", width=0.8)))
fig.add_trace(go.Scatter(x=y_test.index, y=y_test, name="Test",
                          line=dict(color="#E53935", width=0.8)))
fig.update_layout(
    title="Temporal Train / Test Split — test covers all seasons of 2010",
    yaxis_title="Avg Power (kW)", title_x=0.5, height=400
)
fig.show()


## 4. Evaluation Metrics

We report four metrics because they capture different aspects of error:

| Metric | Formula | Why |
|--------|---------|-----|
| **RMSE** | √(mean((y−ŷ)²)) | Penalises large errors heavily; critical for peak events |
| **MAE** | mean(|y−ŷ|) | Average absolute error in kW; interpretable in real units |
| **MAPE** | mean(|y−ŷ|/y)×100 | Percentage error; useful for communicating to non-technical audience |
| **R²** | 1 − SS_res/SS_tot | Fraction of variance explained; 1.0 = perfect, 0 = no better than mean |

A model that only predicts the mean of the training set would score R² = 0.
The naïve baseline scoring R² = −0.14 means it is *worse* than predicting
the mean every hour.


In [ ]:
results = []

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    r2   = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-y_true.mean())**2)
    print(f"[{name:<22}]  RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"MAPE={mape:.1f}%  R²={r2:.4f}")
    return {"Model":name,"RMSE":round(rmse,4),"MAE":round(mae,4),
            "MAPE(%)":round(mape,2),"R2":round(r2,4)}


## 5. Naïve Baseline — Lag-24h Persistence

**What it does:** Predicts tomorrow's 3 PM usage as today's 3 PM usage.
This is the simplest meaningful baseline — it exploits only the daily cycle.

**Why we need it:** Any model that cannot beat this "same time yesterday"
baseline provides zero value over a spreadsheet lookup.

**Result — RMSE: 0.778, R²: −0.14**
The negative R² means the naïve predictor is *worse* than always predicting
the training mean. It fails because actual usage varies day-to-day (weather,
occupancy, special events) in ways that yesterday's reading cannot capture.
This tells us there is real signal to extract — the gap between 0.778 and a
good model's RMSE is what machine learning buys us.


In [ ]:
y_pred_naive = X_test["lag_24h"].values
results.append(evaluate("Naive (lag 24h)", y_test.values, y_pred_naive))

fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test.values[:500], name="Actual",
                          line=dict(color="#1565C0")))
fig.add_trace(go.Scatter(y=y_pred_naive[:500], name="Naive Forecast",
                          line=dict(color="#FB8C00", dash="dash")))
fig.update_layout(
    title="Naïve Baseline (lag-24h) — systematically wrong at peaks and troughs",
    yaxis_title="Avg Power (kW)", title_x=0.5, height=400
)
fig.show()


## 6. Ridge Regression

**What it does:** Linear regression with L2 penalty to prevent overfitting.
All 47 features are scaled to zero-mean unit-variance before fitting.

**Why Ridge before tree models?** Ridge is interpretable and fast. Its
performance sets an upper bound for linear approaches — if Ridge already
performs well, complex models may not be justified.

**Result — RMSE: 0.495, R²: 0.538**
A 36% reduction in RMSE vs the naïve baseline — substantial improvement.
Ridge extracts real signal from the lag features (lag_1h dominates linearly).
However, it fails to capture nonlinear interactions like *"it's winter AND
an evening peak"* → that requires tree-based models.


In [ ]:
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  Ridge(alpha=0.5))
])
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_test)
results.append(evaluate("Ridge Regression", y_test.values, y_pred_ridge))

fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test.values[:500], name="Actual",
                          line=dict(color="#1565C0")))
fig.add_trace(go.Scatter(y=y_pred_ridge[:500], name="Ridge Forecast",
                          line=dict(color="#43A047", dash="dash")))
fig.update_layout(
    title="Ridge Regression — good at averages, misses sharp spikes",
    yaxis_title="Avg Power (kW)", title_x=0.5, height=400
)
fig.show()

joblib.dump(ridge_pipe, "../models/ridge_pipeline.pkl")
print("✅ Ridge model saved")


## 7. Random Forest

**What it does:** Ensemble of 300 decision trees, each trained on a random
subset of features. Predictions are averaged across trees.

**Why Random Forest?** Handles nonlinear interactions naturally. Robust to
outliers. Feature importances are easy to interpret.

**Hyperparameters chosen:**
- `n_estimators=300` — enough trees for stable importance scores
- `max_depth=20` — deep enough to capture interactions, not so deep as to
  overfit training noise
- `max_features=0.6` — 60% of features per split (decorrelates trees)

**Result — RMSE: 0.461, R²: 0.599**
Reduces RMSE by another 7% vs Ridge. The forest correctly learns that
*lag_1h + hour_of_day + season* interact multiplicatively — e.g., a high
lag_1h reading at 8 PM in winter predicts a very high next hour.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=300, max_depth=20, min_samples_leaf=3,
    max_features=0.6, n_jobs=-1, random_state=SEED
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(evaluate("Random Forest", y_test.values, y_pred_rf))

fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test.values[:500], name="Actual",
                          line=dict(color="#1565C0")))
fig.add_trace(go.Scatter(y=y_pred_rf[:500], name="RF Forecast",
                          line=dict(color="#8E24AA", dash="dash")))
fig.update_layout(
    title="Random Forest — captures shape well, slight smoothing at extreme peaks",
    yaxis_title="Avg Power (kW)", title_x=0.5, height=400
)
fig.show()

joblib.dump(rf, "../models/random_forest.pkl")
print("✅ Random Forest saved")


## 8. XGBoost

**What it does:** Gradient boosting — trees built sequentially where each
corrects the previous tree's residuals. Includes early stopping to prevent
overfitting.

**Why XGBoost over Random Forest?** Boosting focuses more on hard examples
(residual errors), which makes it better at capturing peaks.

**Hyperparameters:**
- `learning_rate=0.03` + `n_estimators=1000` with early stopping — slow
  learning rate with many trees is the standard high-performance XGB recipe
- `reg_alpha` and `reg_lambda` — L1 + L2 regularisation to prevent overfit

**Result — RMSE: 0.459, R²: 0.602**
Marginal improvement over Random Forest (0.4% RMSE reduction). The fact that
XGBoost and RF are nearly tied tells us we are approaching the *noise floor*
for this problem — the remaining 0.459 kW RMSE is largely unpredictable
occupancy variation, not a modelling deficiency.


In [ ]:
try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=7,
        min_child_weight=3, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.05, reg_lambda=1.5,
        early_stopping_rounds=30, random_state=SEED, n_jobs=-1, verbosity=0
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred_xgb = xgb_model.predict(X_test)
    results.append(evaluate("XGBoost", y_test.values, y_pred_xgb))

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=y_test.values[:500], name="Actual",
                              line=dict(color="#1565C0")))
    fig.add_trace(go.Scatter(y=y_pred_xgb[:500], name="XGBoost Forecast",
                              line=dict(color="#E53935", dash="dash")))
    fig.update_layout(
        title="XGBoost — closely tracks actual with tighter peak estimates",
        yaxis_title="Avg Power (kW)", title_x=0.5, height=400
    )
    fig.show()
    joblib.dump(xgb_model, "../models/xgboost_model.pkl")
    print("✅ XGBoost saved")
except ImportError:
    print("pip install xgboost")


## 9. LightGBM ← Best Model

**What it does:** Leaf-wise gradient boosting with histogram-based split
finding. Faster training than XGBoost with similar or better accuracy.

**Why LightGBM wins (slightly):**
- Leaf-wise growth allows deeper trees in fewer iterations, capturing the
  nested conditionals in energy data (time × weather × appliance use)
- Histogramming means it handles the 27 k training rows efficiently

**Result — RMSE: 0.459, R²: 0.603 ← Best overall**

The improvement over naive (0.778) represents a **41% RMSE reduction**.
In practical terms: predictions are within ±0.31 kW (MAE) on average —
good enough to identify peak windows and recommend load-shifting strategies.

**Why not perfect?** R² = 0.60 means 40% of variance is unexplained.
This is expected — *occupancy* (is someone home? are they cooking a holiday
meal?) is the dominant driver and is not observed in this dataset.


In [ ]:
try:
    import lightgbm as lgb
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, num_leaves=63, max_depth=8,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.05, reg_lambda=1.5, random_state=SEED, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(
        X_train, y_train, eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )
    y_pred_lgb = lgb_model.predict(X_test)
    results.append(evaluate("LightGBM", y_test.values, y_pred_lgb))

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=y_test.values[:500], name="Actual",
                              line=dict(color="#1565C0")))
    fig.add_trace(go.Scatter(y=y_pred_lgb[:500], name="LightGBM Forecast",
                              line=dict(color="#00897B", dash="dash")))
    fig.update_layout(
        title="LightGBM (Best) — forecast closely follows actual usage pattern",
        yaxis_title="Avg Power (kW)", title_x=0.5, height=400
    )
    fig.show()
    joblib.dump(lgb_model, "../models/lightgbm_model.pkl")
    print("✅ LightGBM saved")
except ImportError:
    print("pip install lightgbm")


## 10. LSTM — Multivariate

**What it does:** A three-layer LSTM network reads 48 consecutive hours
of all 47 features and predicts the next hour.

**Architecture:**
- Layer 1: LSTM(128) + Dropout(0.2)
- Layer 2: LSTM(64) + Dropout(0.2)
- Layer 3: LSTM(32) + Dropout(0.1)
- Dense(16, ReLU) → Dense(1)
- Total parameters: 152,481

**Result — RMSE: 0.589, R²: 0.347 ← Worst ML model**

**Why does LSTM underperform tree models here?**
The loss curves show the LSTM starts overfitting from epoch 3 onward —
validation loss rises while training loss continues to fall. This indicates
the network memorises training patterns without generalising.

Root causes:
1. **Feature-rich tabular data** already encodes all temporal dependencies
   via the 47 lag/rolling features — the LSTM's sequence memory adds little
2. **Only ~27 k training sequences** — deep sequence models typically need
   hundreds of thousands of examples
3. **No exogenous signals** — LSTMs shine when sequences of *external*
   drivers (weather, calendar events) modulate the target; here we have none

**Decision:** Use LightGBM for deployment. The LSTM result is retained as
an important negative finding — deep learning is not always superior.


In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tf.random.set_seed(SEED)
    LOOKBACK = 48

    scaler_lstm = StandardScaler()
    X_lstm_all  = scaler_lstm.fit_transform(df_feat[feature_cols])
    y_lstm_all  = df_feat[TARGET].values

    def make_sequences(X, y, lookback):
        Xs, ys = [], []
        for i in range(lookback, len(X)):
            Xs.append(X[i-lookback:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)

    X_seq, y_seq   = make_sequences(X_lstm_all, y_lstm_all, LOOKBACK)
    split_lstm     = int(len(X_seq) * (1 - TEST_RATIO))
    X_tr, X_te     = X_seq[:split_lstm], X_seq[split_lstm:]
    y_tr, y_te     = y_seq[:split_lstm], y_seq[split_lstm:]

    n_feat = X_seq.shape[2]
    lstm_model = keras.Sequential([
        layers.LSTM(128, input_shape=(LOOKBACK, n_feat), return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.1),
        layers.Dense(16, activation="relu"),
        layers.Dense(1)
    ])
    lstm_model.compile(
        optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    lstm_model.summary()

    cb_es = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True)
    cb_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=0)

    history = lstm_model.fit(
        X_tr, y_tr, epochs=50, batch_size=256,
        validation_split=0.1, callbacks=[cb_es, cb_lr], verbose=1
    )

    y_pred_lstm = lstm_model.predict(X_te).flatten()
    results.append(evaluate("LSTM (multivariate)", y_te, y_pred_lstm))

    # Loss curve
    fig_loss = px.line(
        pd.DataFrame({"Train Loss": history.history["loss"],
                      "Val Loss":   history.history["val_loss"]}),
        title="LSTM — val loss diverges after epoch 3 (overfitting)",
        labels={"value":"MSE Loss","index":"Epoch"},
        color_discrete_sequence=["#1976D2","#E53935"]
    )
    fig_loss.update_layout(title_x=0.5, height=380)
    fig_loss.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=y_te[:500], name="Actual",
                              line=dict(color="#1565C0")))
    fig.add_trace(go.Scatter(y=y_pred_lstm[:500], name="LSTM Forecast",
                              line=dict(color="#FF6F00", dash="dash")))
    fig.update_layout(
        title="LSTM — smoother than actuals; misses sharp spikes",
        yaxis_title="Avg Power (kW)", title_x=0.5, height=400
    )
    fig.show()
    lstm_model.save("../models/lstm_model.keras")
    print("✅ LSTM saved")
except ImportError:
    print("pip install tensorflow")


## 11. Model Comparison

**Summary of all six models sorted by RMSE:**

| Model | RMSE ↓ | MAE ↓ | MAPE% ↓ | R² ↑ |
|-------|--------|-------|---------|------|
| **LightGBM** | **0.459** | **0.314** | **42.1%** | **0.603** |
| XGBoost | 0.459 | 0.316 | 43.6% | 0.602 |
| Random Forest | 0.461 | 0.316 | 43.5% | 0.599 |
| Ridge Regression | 0.495 | 0.349 | 48.9% | 0.538 |
| LSTM (multivariate) | 0.589 | 0.424 | 58.7% | 0.347 |
| Naïve (lag-24h) | 0.778 | 0.520 | 67.6% | −0.140 |

**Key takeaways:**
- LightGBM, XGBoost, and Random Forest are statistically indistinguishable
  (< 1% RMSE difference) — they have hit the noise floor of this dataset
- The gap between gradient boosting and Ridge (8% RMSE) confirms nonlinear
  feature interactions are important
- LSTM's poor performance is a useful result — it rules out deep sequence
  models for this data regime
- **Deployment choice: LightGBM** — fastest inference, best RMSE, native
  feature importance


In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE")
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))


In [ ]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=["RMSE (lower = better)",
                    "MAE  (lower = better)",
                    "R²   (higher = better)"])

palette = ["#0D47A1","#1565C0","#1976D2","#2196F3","#90CAF9","#BBDEFB"]
colors  = palette[:len(results_df)]

for ci, metric in enumerate(["RMSE","MAE","R2"], 1):
    fig.add_trace(
        go.Bar(x=results_df["Model"], y=results_df[metric],
               marker_color=colors, showlegend=False,
               text=[f"{v:.3f}" for v in results_df[metric]],
               textposition="outside"),
        row=1, col=ci
    )
fig.update_layout(title="Model Performance Comparison — LightGBM wins",
                  title_x=0.5, height=520, font=dict(size=10))
fig.show()


## 12. Feature Importance

**Why:** Feature importances tell us *which signals matter* — this feeds
directly into the Streamlit app's energy advisor (e.g., "your most recent
hour of usage is the strongest predictor of the next hour").

### Consistent finding across all three tree models:
1. **`lag_1h`** dominates (RF: 0.43, XGB: 0.31, LGB: 783 splits) —
   the last hour is far and away the strongest predictor
2. **`roll_max_3h`** and **`roll_mean_3h`** rank second/third —
   recent volatility and trend matter
3. **`diff_1h`** (rate of change) ranks top-5 in all models —
   is consumption accelerating or decelerating?
4. **`lag_168h`** (same time last week) outperforms `lag_24h`
   (same time yesterday) in some models — weekly routine is stronger than
   daily routine
5. **Calendar features** (`hour`, `hour_sin`, `hour_cos`) rank in top 10 —
   confirming the strong time-of-day cycle from EDA

**Implication for homeowners:** The single most actionable insight is that
*your consumption right now is the best predictor of your consumption in
the next hour*. Cutting peak-hour loads immediately has compounding benefits.


In [ ]:
# Random Forest importance
rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns)
top_rf  = rf_imp.nlargest(25).sort_values()

fig = go.Figure(go.Bar(
    x=top_rf.values, y=top_rf.index, orientation="h",
    marker_color=BAR_BLUE,
    text=[f"{v:.4f}" for v in top_rf.values], textposition="outside"
))
fig.update_layout(
    title="Top 25 Feature Importances — Random Forest",
    xaxis_title="Importance Score", title_x=0.5, height=650
)
fig.show()


In [ ]:
# XGBoost importance
try:
    xgb_imp = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
    top_xgb  = xgb_imp.nlargest(25).sort_values()
    fig = go.Figure(go.Bar(
        x=top_xgb.values, y=top_xgb.index, orientation="h",
        marker_color="#C62828",
        text=[f"{v:.4f}" for v in top_xgb.values], textposition="outside"
    ))
    fig.update_layout(title="Top 25 Feature Importances — XGBoost",
                      xaxis_title="Importance Score", title_x=0.5, height=650)
    fig.show()
except NameError: pass


In [ ]:
# LightGBM importance
try:
    lgb_imp = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
    top_lgb  = lgb_imp.nlargest(25).sort_values()
    fig = go.Figure(go.Bar(
        x=top_lgb.values, y=top_lgb.index, orientation="h",
        marker_color="#1B5E20",
        text=[f"{v:.0f}" for v in top_lgb.values], textposition="outside"
    ))
    fig.update_layout(title="Top 25 Feature Importances — LightGBM",
                      xaxis_title="Split Count", title_x=0.5, height=650)
    fig.show()
except NameError: pass


## 13. Residual Analysis — LightGBM

**Why:** Residuals (actual − predicted) reveal *systematic errors*. If
residuals correlate with time or magnitude we know the model is still missing
some structure.

**What we find:**
- **Residual distribution:** Approximately normal (bell-shaped) centred near
  zero — no systematic bias
- **Slight right tail:** The model under-predicts extreme high values
  (> 4 kW) — these are rare multi-appliance coincidence events that are
  genuinely hard to predict
- **Actual vs Predicted scatter:** Points cluster near the diagonal but
  scatter increases for higher actuals — heteroscedasticity (typical for
  energy forecasting)
- **Decision:** The model is not systematically biased; the residual
  distribution suggests the remaining error is close to irreducible noise


In [ ]:
try:
    best_pred = y_pred_lgb; best_name = "LightGBM"
except NameError:
    try: best_pred = y_pred_xgb; best_name = "XGBoost"
    except NameError: best_pred = y_pred_rf; best_name = "Random Forest"

residuals = y_test.values - best_pred

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Residual Distribution — centred at 0",
                    "Actual vs Predicted — scatter grows at high loads"])
fig.add_trace(go.Histogram(x=residuals, nbinsx=100, marker_color=BAR_BLUE),
              row=1, col=1)
fig.add_trace(go.Scatter(
    x=y_test.values[:3000], y=best_pred[:3000],
    mode="markers", marker=dict(color=BAR_BLUE, size=3, opacity=0.4)),
    row=1, col=2)
mn, mx = float(y_test.min()), float(y_test.max())
fig.add_trace(go.Scatter(x=[mn,mx], y=[mn,mx], mode="lines",
    line=dict(color="red", dash="dash")), row=1, col=2)
fig.update_layout(
    title=f"{best_name} — Residual Analysis",
    title_x=0.5, height=480, showlegend=False
)
fig.update_xaxes(title_text="Residual (kW)", row=1, col=1)
fig.update_xaxes(title_text="Actual (kW)",   row=1, col=2)
fig.update_yaxes(title_text="Count",         row=1, col=1)
fig.update_yaxes(title_text="Predicted (kW)",row=1, col=2)
fig.show()

print(f"Mean residual (bias): {residuals.mean():.4f} kW  (ideal = 0)")
print(f"Std of residuals    : {residuals.std():.4f} kW")
print(f"% of predictions within ±0.5 kW : "
      f"{(np.abs(residuals) <= 0.5).mean()*100:.1f}%")
print(f"% of predictions within ±1.0 kW : "
      f"{(np.abs(residuals) <= 1.0).mean()*100:.1f}%")


## 14. Walk-Forward Cross-Validation

**Why:** A single train/test split might be unusually easy or hard. Walk-forward
CV mimics sequential deployment — each fold trains on all past data and
tests on the next block.

**What we find (Random Forest, 5 folds):**
- Fold 1 RMSE = 0.636 — hardest fold; model trained on least data
- Folds 2–4 RMSE ≈ 0.520 — stable, consistent generalisation
- Fold 5 RMSE = 0.452 — easiest; model trained on most data, summer period
- **Mean ± Std = 0.529 ± 0.060 kW**

The low standard deviation (0.06) confirms the model is stable across time —
it doesn't collapse or over-fit on any particular period. The improving trend
(fold 1 → fold 5) shows the model gets better with more training data.


In [ ]:
tscv   = TimeSeriesSplit(n_splits=5)
rf_cv  = RandomForestRegressor(n_estimators=100, max_depth=15,
                                min_samples_leaf=3, max_features=0.6,
                                n_jobs=-1, random_state=SEED)
cv_raw = cross_val_score(rf_cv, X, y, cv=tscv,
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
cv_rmse = -cv_raw

print("Walk-Forward CV RMSE per fold:")
for i,s in enumerate(cv_rmse, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"  Mean ± Std : {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")

fig = go.Figure(go.Bar(
    x=[f"Fold {i}" for i in range(1,6)], y=cv_rmse,
    marker_color=BAR_BLUE,
    text=[f"{v:.4f}" for v in cv_rmse], textposition="outside"
))
fig.add_hline(y=cv_rmse.mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean = {cv_rmse.mean():.4f}")
fig.update_layout(title="Walk-Forward CV RMSE — stable across time",
                  yaxis_title="RMSE (kW)", title_x=0.5, height=420)
fig.show()


## 15. Error Analysis — By Hour and Season

**Why:** Knowing *when* the model is least accurate tells homeowners which
forecasts to trust and helps identify targets for future improvement.

### By Hour
- **Night (0–5 AM): MAE ≈ 0.15–0.20 kW** — easiest; low, stable standby loads
- **Morning (6–9 AM): MAE spikes to 0.38 kW** — breakfast/shower routines are
  highly variable (some days people cook, other days they don't)
- **Evening (19–22h): MAE = 0.42–0.49 kW** — hardest; dinner + TV + arbitrary
  appliance combinations are the most variable

### By Season
- **Summer: MAE = 0.26 kW** — easiest; air-conditioning follows a predictable
  temperature-driven pattern
- **Winter: MAE = 0.38 kW** — hardest; heating behaviour is more variable
  (thermostat settings, number of people home, cooking habits vary day-to-day)

**Homeowner implication:** Forecasts at 8 AM and 8 PM should be treated with
more uncertainty than overnight forecasts. In winter, add ±20% to the model's
prediction interval.


In [ ]:
error_df = pd.DataFrame({
    "actual"  : y_test.values,
    "pred"    : best_pred,
    "abs_err" : np.abs(y_test.values - best_pred),
    "hour"    : y_test.index.hour,
    "season"  : y_test.index.month.map({
        12:"Winter",1:"Winter",2:"Winter",
        3:"Spring", 4:"Spring",5:"Spring",
        6:"Summer", 7:"Summer",8:"Summer",
        9:"Autumn",10:"Autumn",11:"Autumn"})
}, index=y_test.index)

# Hour of day
h_err = error_df.groupby("hour")["abs_err"].mean().reset_index()
fig = go.Figure(go.Bar(x=h_err["hour"], y=h_err["abs_err"],
                        marker_color=BAR_BLUE))
fig.update_layout(
    title="MAE by Hour of Day — evening peaks (19–22h) hardest to predict",
    xaxis_title="Hour of Day", yaxis_title="MAE (kW)",
    title_x=0.5, height=420
)
fig.show()


In [ ]:
# Season
s_err = error_df.groupby("season")["abs_err"].mean()    .reindex(["Winter","Spring","Summer","Autumn"]).reset_index()
fig = go.Figure(go.Bar(x=s_err["season"], y=s_err["abs_err"],
                        marker_color=BAR_BLUE,
                        text=[f"{v:.3f}" for v in s_err["abs_err"]],
                        textposition="outside"))
fig.update_layout(
    title="MAE by Season — summer easiest (0.26 kW), winter hardest (0.38 kW)",
    xaxis_title="Season", yaxis_title="MAE (kW)",
    title_x=0.5, height=420
)
fig.show()


## 16. Quantile Performance Analysis

**Why:** Average metrics hide performance at different consumption levels.
A model might be excellent for mid-range loads but poor at extremes.
This matters for demand-response systems that trigger on *peaks*.

**What we find:**
- Low consumption (< 0.5 kW, Q25): relative error is higher (MAPE ~35–50%)
  but absolute error is small — acceptable
- Mid consumption (0.5–2.0 kW): best MAE, lowest relative error
- High consumption (> 3.0 kW, top 10%): MAE rises sharply — these peak
  events are least predictable but most important for grid management

**Implication:** For a demand-response application, augmenting the model with
real-time temperature data or a "people-at-home" signal would most improve
peak-period accuracy.


In [ ]:
quantiles = [0, 0.25, 0.5, 0.75, 0.9, 1.0]
labels    = ["Q0–25", "Q25–50", "Q50–75", "Q75–90", "Q90–100"]
bins      = pd.qcut(error_df["actual"], q=quantiles, labels=labels)
q_err     = error_df.groupby(bins)["abs_err"].agg(["mean","median","count"])
q_err.columns = ["MAE","Median_AE","Count"]
q_err["Range_kW"] = [f"{error_df['actual'].quantile(quantiles[i]):.2f}–"
                     f"{error_df['actual'].quantile(quantiles[i+1]):.2f}"
                     for i in range(len(labels))]
print("Error by consumption quantile:")
print(q_err.to_string())

fig = go.Figure(go.Bar(
    x=q_err.index.astype(str), y=q_err["MAE"],
    text=[f"{v:.3f}" for v in q_err["MAE"]],
    textposition="outside",
    marker_color=BAR_BLUE
))
fig.update_layout(
    title="MAE by Consumption Quantile — errors grow at high loads (peak events)",
    xaxis_title="Consumption Quantile", yaxis_title="MAE (kW)",
    title_x=0.5, height=420
)
fig.show()


## 17. Save Models & Feature Columns


In [ ]:
# Save feature columns for use by the Streamlit app
import json
with open("../models/feature_cols.json","w") as f:
    json.dump(feature_cols, f)

print("✅ All models saved to ../models/")
print("   ridge_pipeline.pkl | random_forest.pkl")
print("   xgboost_model.pkl  | lightgbm_model.pkl")
print("   lstm_model.keras   | feature_cols.json")


## 18. Final Conclusions & Recommendations

### Results Summary
| Model | RMSE | R² |
|---|---|---|
| **LightGBM** ← deploy | **0.459** | **0.603** |
| XGBoost | 0.459 | 0.602 |
| Random Forest | 0.461 | 0.599 |
| Ridge Regression | 0.495 | 0.538 |
| LSTM | 0.588 | 0.349 |
| Naïve baseline | 0.778 | −0.140 |

**41% RMSE reduction** vs naïve baseline. 80% of predictions fall within ±0.5 kW.

### Key Findings
- **lag_1h dominates** all three tree models — the last hour of usage is the strongest signal
- **LightGBM ≈ XGBoost ≈ Random Forest** (< 1% RMSE gap) — the models have hit the dataset's noise floor
- **LSTM underperforms** tree models; the 47 engineered features already encode temporal structure, leaving nothing for the sequence memory to add
- **R² = 0.60** means ~40% of variance is unexplained — primarily unobserved occupancy behaviour

### Why It's Not Better
No occupancy signal, no weather data. These two factors alone likely account for the majority of the residual 0.46 kW RMSE.

### Actionable Recommendations
- **Evening (19–22h)** forecasts carry the highest uncertainty — add ±30% buffer
- **Winter** MAE is 44% higher than summer — widen prediction intervals seasonally
- **If usage is high right now**, expect it to remain high — shift discretionary loads immediately
- Sub-metering the main circuits would be the single highest-ROI data improvement